# 2. Is the extracted text the right document?


In [ ]:
%pip install -q "pandas>=2.0"

import glob, os, zipfile
import pandas as pd

RUN_FROM_SCRATCH = False
WRDS_USER = ""

print("pandas", pd.__version__)


In [ ]:
import importlib
import os
import sys
from pathlib import Path

_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts" / "project_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside S08, or set S08_ROOT to the checkout path.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import project_paths
importlib.reload(project_paths)
from project_paths import (REPO_ROOT, TEAM_DIR, SCRIPTS, NOV2_DATA,
                           PROJECT_DIR, STEP5_DIR, PART34_CODE, STEP5_CODE)

import os

DATA = str(NOV2_DATA)

NEED = [
    "attrition_funnel.csv",
    "raw_counts.csv",
    "matched_sample.csv",
    "extraction_report.csv",
    "timing_validation.csv",
    "wire_times.csv",
    "three_source_timing.csv",
    "event_table.csv"
]

RESULTS = str(TEAM_DIR / 'results/2_results')
os.makedirs(RESULTS, exist_ok=True)

#check that all required files are available
missing = [
    f for f in NEED
    if not os.path.isfile(os.path.join(DATA, f))
    or os.path.getsize(os.path.join(DATA, f)) == 0
]

if missing:
    raise FileNotFoundError(f"Missing inputs in {DATA}: {missing}. Restore the private data or set TEAM_DIR.")
else:
    print("All required files found.")

#manual validation folder
BASE = str(TEAM_DIR / '2_EDGAR text')
MANUAL = os.path.join(BASE, "Manual validation")
os.makedirs(MANUAL, exist_ok=True)

#list available data files
for f in sorted(os.listdir(DATA)):
    path = os.path.join(DATA, f)
    if os.path.isfile(path):
        print(f"{os.path.getsize(path):>12,}  {f}")

In [ ]:
#extraction report

import re

report = pd.read_csv(f"{DATA}/extraction_report.csv",
                     dtype={"accession": str})
report["accession"] = report.accession.str.strip()
report["year"] = pd.to_datetime(report.filing_date, errors="coerce").dt.year

print(f"{len(report):,} filings")
print(report.source.value_counts().head(8).to_string())

In [ ]:
#EDGAR extraction

import html, threading, time, urllib.error, urllib.request

USER_AGENT = "MIT 15.S08 project mbonnetderoovere501@gmail.com"
RATE = 9.0
WORKERS = 8
MIN_WORDS = 500

ROW_RE = re.compile(r"<tr[^>]*>(.*?)</tr>", re.S | re.I)
CELL_RE = re.compile(r"<td[^>]*>(.*?)</td>", re.S | re.I)
HREF_RE = re.compile(r'href="([^"]+)"', re.I)
SCRIPT_RE = re.compile(r"<(script|style)\b.*?</\1>", re.S | re.I)
TABLE_RE = re.compile(r"<table\b.*?</table>", re.S | re.I)
TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"[ \t\r\f\v]+")
NL_RE = re.compile(r"\n{3,}")


class RateLimiter:

    def __init__(self, rate: float):
        self.min_gap = 1.0 / rate
        self.lock = threading.Lock()
        self.next_at = 0.0

    def acquire(self) -> None:
        with self.lock:
            now = time.monotonic()
            wait = max(0.0, self.next_at - now)
            self.next_at = max(now, self.next_at) + self.min_gap
        if wait:
            time.sleep(wait)

def fetch(url: str, limiter: RateLimiter, retries: int = 4) -> bytes | None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    for attempt in range(retries):
        limiter.acquire()
        try:
            with urllib.request.urlopen(req, timeout=45) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code in (403, 429, 500, 502, 503, 504):
                time.sleep(min(2 ** attempt, 20))
                continue
            return None
        except Exception:
            time.sleep(min(2 ** attempt, 20))
    return None

def _strip(fragment: str) -> str:
    return html.unescape(TAG_RE.sub(" ", fragment)).replace("\xa0", " ")


def to_text(raw: str) -> tuple[str, int]:
    body = SCRIPT_RE.sub(" ", raw)

    dropped = 0

    def decide(m: re.Match) -> str:
        nonlocal dropped
        inner = _strip(m.group(0))
        letters = sum(c.isalpha() for c in inner)
        digits = sum(c.isdigit() for c in inner)
        if digits > 0.30 * max(letters + digits, 1):
            dropped += 1
            return " "
        return " " + inner + " "

    body = TABLE_RE.sub(decide, body)
    body = _strip(body)
    body = NL_RE.sub("\n\n", WS_RE.sub(" ", body))
    return body.strip(), dropped


def looks_like_prose(text: str) -> bool:
    words = text.split()
    if len(words) < MIN_WORDS or text.count(". ") < 10:
        return False
    return sum(c.isdigit() for c in text) / max(len(text), 1) < 0.25


def rank(doc_type: str) -> int:
    t = doc_type.upper()
    if t == "EX-99.1":
        return 0
    if t == "EX-99.2":
        return 1
    if t.startswith("EX-99"):
        return 2
    if t == "8-K":
        return 3
    return 9


def parse_index(body: str, base: str) -> list[tuple[str, str]]:
    out = []
    for row in ROW_RE.findall(body):
        cells = CELL_RE.findall(row)
        if len(cells) < 4:
            continue
        doc_type = re.sub(r"<[^>]+>", " ", cells[3]).strip()
        if rank(doc_type) == 9:
            continue
        m = HREF_RE.search(cells[2])
        if not m:
            continue
        href = html.unescape(m.group(1))
        if "ix?doc=" in href:
            href = href.split("ix?doc=", 1)[1]
        url = href if href.startswith("http") else \
            "https://www.sec.gov" + (href if href.startswith("/") else "/" + href)
        out.append((doc_type, url))
    out.sort(key=lambda p: rank(p[0]))
    return out


def one_filing(row: dict, limiter: RateLimiter) -> tuple[dict, str | None]:
    acc = row["accession"]
    cik = str(row["cik"]).lstrip("0")
    base = (f"https://www.sec.gov/Archives/edgar/data/{cik}/"
            f"{acc.replace('-', '')}/")
    rec = {"accession": acc, "cik": cik,
           "filing_date": row.get("filing_date", ""),
           "source": "", "filename": "", "words": 0, "tables_dropped": 0}

    idx = fetch(base + f"{acc}-index.htm", limiter)
    if idx is None:
        rec["source"] = "index fetch failed"
        return rec, None

    for doc_type, url in parse_index(idx.decode("utf-8", "replace"), base):
        body = fetch(url, limiter)
        if body is None:
            continue
        text, n_tables = to_text(body.decode("utf-8", "replace"))
        if looks_like_prose(text):
            rec["source"] = doc_type if doc_type.upper().startswith("EX-99") \
                else "8-K body"
            rec["filename"] = url.rsplit("/", 1)[-1]
            rec["words"] = len(text.split())
            rec["tables_dropped"] = n_tables
            return rec, text

    rec["source"] = "none"
    return rec, None

print("extractor loaded")

In [ ]:
#sample selection

def family(s):
    s = str(s).strip().upper()
    if s == "8-K BODY":
        return "8-K body"
    m = re.match(r"EX-99[.,`-]*0*(\d+)", s)
    if m:
        return "EX-99.1" if int(m.group(1)) == 1 else "EX-99.n"
    return "EX-99 other"


pool = report[report.year.between(2006, 2025) & ~report.source.isin(["none", "index fetch failed"])].copy()

pool["family"] = pool.source.map(family)
print(pool.family.value_counts().to_string(), "\n")

main = pool.groupby("year", group_keys=False).sample(n=5, random_state=20261104).assign(stratum="main")

rare_types = pool.family.value_counts(normalize=True)
rare = pd.concat([pool[pool.family == f].sample(n=min(8, (pool.family == f).sum()), random_state=20261104)
    for f in rare_types[rare_types < 0.02].index
]).assign(stratum="rare")

sample = pd.concat([main, rare])
sample = sample.drop_duplicates("accession")
sample = sample.sort_values(["stratum", "year", "accession"]).reset_index(drop=True)

assert (sample.stratum == "main").sum() == 100

print(f"{len(sample)} filings: {(sample.stratum == 'main').sum()} main, "
      f"{(sample.stratum == 'rare').sum()} rare")
print(sample.groupby(["stratum", "family"]).size().to_string())

In [ ]:
#check extraction

limiter = RateLimiter(RATE)

FETCHED = {}
_fetch = fetch

def fetch(url, limiter, retries=4):
    body = _fetch(url, limiter, retries)
    if body is not None:
        FETCHED[url] = body
    return body


def to_text_keep_tables(raw):
    body = _strip(SCRIPT_RE.sub(" ", raw))
    return NL_RE.sub("\n\n", WS_RE.sub(" ", body)).strip(), 0


texts, raw_html, checks = {}, {}, []

for i, row in sample.iterrows():
    rec, text = one_filing({"accession": row.accession, "cik": row.cik, "filing_date": row.filing_date}, limiter)

    texts[row.accession] = text or ""

    if rec["filename"]:
        for url, body in FETCHED.items():
            if url.endswith("/" + rec["filename"]):
                raw_html[row.accession] = body.decode("utf-8", "replace")
                break

    checks.append({
        "accession": row.accession,
        "source_recorded": row.source,
        "source_now": rec["source"],
        "words_recorded": int(row.words),
        "words_now": int(rec["words"]),
        "same_exhibit": rec["source"] == row.source,
        "same_words": int(rec["words"]) == int(row.words)
    })

    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(sample)}", flush=True)

chk = pd.DataFrame(checks)
agree = chk.same_exhibit & chk.same_words

print(f"\nreproduces the recorded extraction: {agree.sum()}/{len(chk)}")

if (~agree).any():
    print("\ndoes not reproduce:")
    print(chk[~agree].to_string(index=False))

chk.to_csv(os.path.join(RESULTS, "extraction_reproduction.csv"), index=False)

In [ ]:
#table removal

NUM_RE = re.compile(r"\b\d[\d,.]*%?\b")

loss = []

for acc, kept in texts.items():
    raw = raw_html.get(acc)
    if not raw or not kept:
        continue

    whole, _ = to_text_keep_tables(raw)
    w_whole, w_kept = whole.split(), kept.split()

    n_whole = len(NUM_RE.findall(whole))
    n_kept = len(NUM_RE.findall(kept))

    prose_whole = sum(not NUM_RE.fullmatch(w) for w in w_whole)
    prose_kept = sum(not NUM_RE.fullmatch(w) for w in w_kept)

    loss.append({
        "accession": acc,
        "words_with_tables": len(w_whole),
        "words_kept": len(w_kept),
        "numbers_with_tables": n_whole,
        "numbers_kept": n_kept,
        "prose_with_tables": prose_whole,
        "prose_kept": prose_kept,
        "pct_numbers_dropped": round(100 * (n_whole - n_kept) / max(n_whole, 1), 1),
        "pct_prose_dropped": round(100 * (prose_whole - prose_kept) / max(prose_whole, 1), 1)
    })

ls = pd.DataFrame(loss)

print(f"{len(ls)} filings compared with and without the table rule\n")
print(f"numbers removed   median {ls.pct_numbers_dropped.median():5.1f}%  "
      f"mean {ls.pct_numbers_dropped.mean():5.1f}%")
print(f"narrative removed median {ls.pct_prose_dropped.median():5.1f}%  "
      f"mean {ls.pct_prose_dropped.mean():5.1f}%")

suspect = ls[ls.pct_prose_dropped > 40].sort_values("pct_prose_dropped", ascending=False)

print(f"\nlost more than 40% of their narrative: {len(suspect)}")

if len(suspect):
    print(suspect[["accession", "pct_prose_dropped", "words_kept"]].head(10).to_string(index=False))
    print("\nThese are the ones to read first in the sheet below: if the")
    print("rule ate the release, text_usable is n and the fix is in to_text.")

ls.to_csv(os.path.join(RESULTS, "table_rule_loss.csv"), index=False)
print("\n->", os.path.join(RESULTS, "table_rule_loss.csv"))

In [ ]:
#Manual validation

import html

MANUAL = str(TEAM_DIR / '2_EDGAR text/Manual validation')
os.makedirs(MANUAL, exist_ok=True)

SHEET = os.path.join(MANUAL, "extraction_handcheck.csv")

def edgar(cik, acc):
    return (f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/"
            f"{acc.replace('-', '')}/{acc}-index.htm")


sheet = pd.DataFrame({
    "n": range(1, len(sample) + 1),
    "stratum": sample.stratum.values,
    "year": sample.year.values,
    "accession": sample.accession.values,
    "exhibit_taken": sample.source.values,
    "family": sample.family.values,
    "filename": sample.filename.values,
    "words": sample.words.values,
    "tables_dropped": sample.tables_dropped.values,
    "edgar_index": [edgar(c, a) for c, a in zip(sample.cik, sample.accession)],
    "right_exhibit": "",
    "is_earnings_release": "",
    "text_usable": "",
    "note": ""
})

sheet.to_csv(SHEET, index=False, encoding="utf-8-sig")

parts = [
    "<html><head><meta charset='utf-8'><style>"
    "body{font-family:Georgia,serif;margin:40px;max-width:900px}"
    "section{border-top:1px solid #ccc;padding-top:20px;margin-top:40px}"
    "pre{white-space:pre-wrap;background:#f6f6f6;padding:16px;"
    "max-height:520px;overflow:auto;font-family:monospace;font-size:13px}"
    "</style></head><body><h1>Extraction hand check</h1>"
    "<p>For each filing: open it on EDGAR, read what we extracted, and "
    "answer the three questions in the sheet.</p>"
]

for _, r in sheet.iterrows():
    t = texts.get(r.accession, "")
    parts.append(
        f"<section><h2>{r.n}. {r.accession} &middot; {r.year} "
        f"&middot; {r.stratum}</h2>"
        f"<p>we took <b>{html.escape(str(r.exhibit_taken))}</b> "
        f"({html.escape(str(r.filename))}), {r.words:,} words, "
        f"{r.tables_dropped} tables dropped &middot; "
        f"<a href='{r.edgar_index}' target='_blank'>open on EDGAR</a></p>"
        f"<pre>{html.escape(t) if t else 'NOTHING EXTRACTED'}</pre></section>"
    )

parts.append("</body></html>")

page = os.path.join(MANUAL, "extraction_handcheck.html")

with open(page, "w", encoding="utf-8") as f:
    f.write("\n".join(parts))

print("sheet  ->", SHEET)
print("viewer ->", page)

In [ ]:
#cleaned texts

cols = ["accession", "acceptance_et", "session_date", "when"]

head = pd.read_csv(f"{DATA}/event_table.csv", nrows=0).columns
if "release_et" in head:
    cols.insert(2, "release_et")

ev = pd.read_csv(f"{DATA}/event_table.csv", dtype={"accession": str}, usecols=cols).drop_duplicates("accession")

ds = pd.DataFrame({"accession": list(texts), "text": [texts[a] for a in texts]})

ds["words"] = ds.text.str.split().str.len()

ds = (ds.merge(sample[["accession", "source", "filename", "family", "tables_dropped"]], on="accession", how="left").merge(ev, on="accession", how="left"))

ds = ds[[
    c for c in [
        "accession", "acceptance_et", "release_et", "session_date",
        "when", "source", "family", "filename",
        "tables_dropped", "words", "text"
    ] if c in ds.columns
]]

path = os.path.join(RESULTS, "release_texts.csv")
ds.to_csv(path, index=False, encoding="utf-8")

print(f"{len(ds):,} texts, {ds.words.sum():,} words")
print(f"with a timestamp: {ds.acceptance_et.notna().sum():,}")
print("columns:", ", ".join(ds.columns))
print("->", path)

In [ ]:
#automatic validation

EARNINGS = [
    "earnings per share", "net income", "diluted", "fourth quarter",
    "first quarter", "second quarter", "third quarter", "full year",
    "results of operations", "financial results", "net revenue",
    "operating income", "gaap", "guidance"
]

NOT_EARNINGS = [
    "traffic", "revenue passenger", "load factor",
    "available seat miles", "same-store sales for",
    "monthly sales", "sales for the month", "unit sales for"
]


def score_text(t):
    low = t[:8000].lower()
    return (sum(k in low for k in EARNINGS), sum(k in low for k in NOT_EARNINGS))

auto = []

for _, row in sample.iterrows():
    acc = row.accession
    text = texts.get(acc, "")

    if not text:
        auto.append({
            "accession": acc,
            "right_exhibit_auto": "n",
            "is_earnings_release_auto": "n",
            "text_usable_auto": "n",
            "auto_note": "nothing extracted"
        })
        continue

    hits, anti = score_text(text)

    cik = str(int(row.cik))
    base = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc.replace('-', '')}/"

    best_other, best_name = -1, ""
    idx = fetch(base + f"{acc}-index.htm", limiter)

    if idx is not None:
        for doc_type, url in parse_index(idx.decode("utf-8", "replace"), base):
            name = url.rsplit("/", 1)[-1]

            if name == row.filename:
                continue

            body = fetch(url, limiter)
            if body is None:
                continue

            other, _ = to_text(body.decode("utf-8", "replace"))
            if not looks_like_prose(other):
                continue

            h, a = score_text(other)
            if h - a > best_other:
                best_other, best_name = h - a, name

    lost = ls.loc[ls.accession == acc, "pct_prose_dropped"]
    lost = float(lost.iloc[0]) if len(lost) else 0.0

    right = "y" if hits - anti >= best_other else "n"
    release = "y" if hits >= 3 and anti <= 1 else "n"
    usable = "y" if lost <= 40 and len(text.split()) >= 300 else "n"

    note = []
    if right == "n":
        note.append(f"{best_name} scores higher")
    if release == "n":
        note.append(f"earnings words {hits}, traffic words {anti}")
    if usable == "n":
        note.append(f"{lost:.0f}% of the narrative dropped, "
                    f"{len(text.split()):,} words left")

    auto.append({
        "accession": acc,
        "right_exhibit_auto": right,
        "is_earnings_release_auto": release,
        "text_usable_auto": usable,
        "auto_note": "; ".join(note)
    })

au = pd.DataFrame(auto)

for c in ["right_exhibit", "is_earnings_release", "text_usable"]:
    print(f"{c:<22} n = {(au[c + '_auto'] == 'n').sum():>3}")

sheet2 = pd.read_csv(SHEET).merge(au, on="accession", how="left")

for c in ["right_exhibit", "is_earnings_release", "text_usable"]:
    sheet2[c] = sheet2[c + "_auto"]

sheet2["judged_by"] = "code"
sheet2.to_csv(SHEET, index=False, encoding="utf-8-sig")

print("\nsheet refilled with the machine pass ->", SHEET)

## Manual validation

We also need to manually check the extracted texts, since the automatic checks might miss some errors.

For this, we open `extraction_handcheck.html` and compare each extracted text with the original filing on EDGAR. We then check three things:

- **`right_exhibit`**: Did we select the correct exhibit? Some filings have several exhibits, so we need to make sure we extracted the earnings release.
- **`is_earnings_release`**: Is the document actually an earnings release? For example, some Item 2.02 filings contain monthly sales or traffic reports, which we do not want in our sample.
- **`text_usable`**: Is the extracted text usable? We mainly want to check that removing financial tables did not also remove important parts of the text, or leave too much irrelevant information.

The CSV already contains the answers from the automatic checks. We go through them and change `y` to `n` (or the opposite) when we disagree. We can also add a short explanation in the `note` column when something looks wrong.

The idea is to calculate an accuracy rate, but also to see what types of mistakes the extraction code makes and whether we need to improve it.

In [ ]:
#extraction accuracy

filled = pd.read_csv(SHEET)
cols = ["right_exhibit", "is_earnings_release", "text_usable"]

def yes(s):
    return s.astype(str).str.strip().str.lower().isin(["y", "yes", "true", "1"])

def no(s):
    return s.astype(str).str.strip().str.lower().isin(["n", "no", "false", "0"])


answered = filled[cols].apply(yes) | filled[cols].apply(no)
done = filled[answered.all(axis=1)]

#we check if we changed any of the automatic answers
auto_cols = [c + "_auto" for c in cols]

if all(c in filled.columns for c in auto_cols):
    changed = (filled[cols].values != filled[auto_cols].values).any(axis=1)
    n_human = int(changed.sum())

    print(f"judged: {len(done)}/{len(filled)}   "
          f"{n_human} rows corrected by hand, "
          f"{len(filled) - n_human} as the code left them")

    if n_human == 0:
        print("This is the machine pass. The rate below is provisional "
              "until the sheet is read.")
else:
    print(f"judged: {len(done)}/{len(filled)}")


lines = []

if len(done) > 0:
    done = done.assign(correct=done[cols].apply(yes).all(axis=1))

    #we calculate accuracy separately for the main and rare samples
    for block in ["main", "rare"]:
        b = done[done.stratum == block]

        if len(b) == 0:
            continue

        accuracy = 100 * b.correct.mean()
        label = ("extraction accuracy" if block == "main"
                 else "rare exhibit types, reported separately")

        print(f"\n{label}: {accuracy:.1f}%  "
              f"({b.correct.sum()}/{len(b)})")

        lines.append({
            "block": block,
            "n": len(b),
            "correct": int(b.correct.sum()),
            "accuracy_pct": round(accuracy, 1)
        })

        for c in cols:
            bad = no(b[c])
            print(f"  fails {c:<22} {bad.sum():>3}  {100 * bad.mean():5.1f}%")

    print("\nby exhibit family:")
    print(done.groupby(["stratum", "family"]).correct
          .agg(["size", "mean"]).to_string())

    print("\nnotes on the failures:")
    for _, r in done[~done.correct].iterrows():
        why = r.get("note")

        if not isinstance(why, str) or not why.strip():
            why = r.get("auto_note", "")

        print(f"  {r.stratum:<5} {r.accession}  {r.exhibit_taken:<10} {why}")

    pd.DataFrame(lines).to_csv(os.path.join(RESULTS, "extraction_accuracy.csv"), index=False)
    done.to_csv(os.path.join(RESULTS, "extraction_handcheck_judged.csv"), index=False)

    print("\n->", os.path.join(RESULTS, "extraction_accuracy.csv"))

else:
    print("\nFill extraction_handcheck.csv in the Manual validation folder,")
    print("then run this cell again. Three columns a row, y or n.")

In [ ]:
#checkig possible extraction errors

#first, we identify potential problems without manual validation
print("WHAT THE CODE CAN SEE WITHOUT A READER\n")

ate = ls[ls.pct_prose_dropped > 40]

print(f"{len(ate):>3}  the table rule removed more than 40% of the "
      f"narrative")
print("     to_text judges a table by its digit ratio and drops it above "
      "0.30.")
print("     A release laid out as one table goes out whole. Of the three")
print("     failure modes this is the only one the code can prove alone.")

short = sample[sample.words < 300]

print(f"\n{len(short):>3}  fewer than 300 words kept")
print("     looks_like_prose accepts at 500 words before the table rule")
print("     runs, so anything this short lost its body afterwards.")

multi = report[report.accession.isin(sample.accession)]
odd = multi[~multi.source.astype(str).str.upper().str.match(r"EX-99[.,`-]*0*1")]

print(f"\n{len(odd):>3}  an exhibit other than EX-99.1 was taken")
print("     rank() tries EX-99.1 first, so these are filings where it was")
print("     absent or failed looks_like_prose. Not wrong, but the place")
print("     where taking the wrong document is most likely.")

flagged = sorted(set(ate.accession) | set(short.accession) | set(odd.accession))

print(f"\n{len(flagged)} filings flagged by at least one of the three. "
      f"Read these first.")

pd.DataFrame({"accession": flagged}).to_csv(os.path.join(RESULTS, "extraction_flagged.csv"), index=False)


#we then check the errors found during manual validation
print("\n" + "-" * 68)
print("WHAT NEEDS THE SHEET\n")

if len(done) == 0 or done.correct.all():
    print("Nothing judged yet, or nothing judged has failed. The three")
    print("counts above stand on their own; the split below needs the")
    print("sheet, because only a reader can say whether the document is")
    print("the earnings release.")

else:
    bad = done[~done.correct].merge(ls[["accession", "pct_prose_dropped", "pct_numbers_dropped"]], on="accession", how="left")

    buckets = {
        "rank / parse_index: took the wrong exhibit":
            no(bad.right_exhibit),
        "looks_like_prose: accepted a document that is not a release":
            no(bad.is_earnings_release),
        "to_text: the table rule damaged the text":
            no(bad.text_usable)
    }

    for label, mask in buckets.items():
        k = int(mask.sum())
        share = 100 * k / max(len(done), 1)

        if share >= 3:
            verdict = "systematic, worth changing the rule"
        else:
            verdict = "too few to call systematic"

        print(f"{k:>3}  {share:4.1f}% of the sample   {label}")
        print(f"     {verdict}")

    ate = bad[(bad.pct_prose_dropped > 40) & no(bad.text_usable)]

    if len(ate) > 0:
        print(f"\n{len(ate)} of the damaged texts also lost more than 40% of")
        print("their narrative to the table rule. That is one cause, not two:")
        print("the release was laid out as a table and to_text dropped it on")
        print("its digit ratio. The threshold is the 0.30 in to_text.")

    bad.to_csv(os.path.join(RESULTS, "extraction_failures.csv"), index=False)

    print("\n->", os.path.join(RESULTS, "extraction_failures.csv"))

In [ ]:
#figures results

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

v = report.source.map(family).value_counts()
ax[0].barh(v.index[::-1], v.values[::-1], color="#4a5568")
ax[0].set_title("Which exhibit the extractor took")
ax[0].set_xlabel("filings")

for i, n in enumerate(v.values[::-1]):
    ax[0].text(n, i, f" {n:,}", va="center", fontsize=9)

#numbers removed
ax[1].hist(ls.pct_numbers_dropped, bins=30, color="#4a5568")
ax[1].set_title("Numbers removed with the tables")
ax[1].set_xlabel("% of numeric tokens dropped")
ax[1].set_ylabel("filings")

#text removed
ax[2].hist(ls.pct_prose_dropped, bins=30, color="#9b2c2c")
ax[2].axvline(40, color="black", lw=1, ls="--")
ax[2].set_title("Narrative removed with the tables")
ax[2].set_xlabel("% of non-numeric words dropped")

for a in ax:
    a.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
p1 = os.path.join(RESULTS, "extraction_quality.png")
fig.savefig(p1, dpi=150)
print("->", p1)

#accuracy
if len(done) > 0 and "correct" in done:
    fig2, ax2 = plt.subplots(figsize=(6, 4))

    g = done.groupby("family").correct.mean().sort_values() * 100
    ax2.barh(g.index, g.values, color="#2c5282")
    ax2.set_xlim(0, 100)
    ax2.set_xlabel("% correct")
    ax2.set_title("Extraction accuracy by exhibit family")

    for i, n in enumerate(g.values):
        ax2.text(n, i, f" {n:.0f}%", va="center", fontsize=9)

    ax2.spines[["top", "right"]].set_visible(False)
    fig2.tight_layout()

    p2 = os.path.join(RESULTS, "extraction_accuracy.png")
    fig2.savefig(p2, dpi=150)
    print("->", p2)

else:
    print("accuracy chart skipped: the sheet is not filled yet")